In [ ]:
import sys
import os

# 1. DRIVE MOUNT AND PATHS SETUP
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    DRIVE_ROOT = '/content/drive/MyDrive/AA-STAL/data_pipeline'

    if DRIVE_ROOT not in sys.path:
        sys.path.append(DRIVE_ROOT)

    print(f"Directory: {os.getcwd()}")
except ImportError:
    # Fallback for local execution
    DRIVE_ROOT = '.'
    print("Local execution")

In [ ]:
# Copy archive
!cp /content/drive/MyDrive/AA-STAL/data_pipeline/DATA_ROOT/DATA_ROOT.tar.gz /content/

# Extract
!tar -xf /content/DATA_ROOT.tar.gz -C /content/

# Delete archive
!rm /content/DATA_ROOT.tar.gz

In [ ]:
!cp -r /content/drive/MyDrive/AA-STAL/data_pipeline/DATA_ROOT/groundtruth/PennAction/ /content/

In [ ]:
import os
import json
import glob
import re
import csv
import shutil
import random
from collections import Counter, defaultdict
from PIL import Image

# PATHS
BASE_DIR = "/content/"

VIDEOS_DIR = os.path.join(BASE_DIR, "Videos_crop")
FRAMES_DIR = os.path.join(BASE_DIR, "Videos_crop_decode")
DETECTION_DIR = os.path.join(BASE_DIR, "video_general_obj_det_finished")
ACTION_DIR = os.path.join(BASE_DIR, "action_recognition_finished")

AVA_ROOT = os.path.join(BASE_DIR, "AVA_Dataset")
OUT_ANNOTATIONS = os.path.join(AVA_ROOT, "annotations")
OUT_FRAMES = os.path.join(AVA_ROOT, "frames")
OUT_FRAMES_LISTS = os.path.join(AVA_ROOT, "frame_lists")

os.makedirs(OUT_ANNOTATIONS, exist_ok=True)
os.makedirs(OUT_FRAMES, exist_ok=True)
os.makedirs(OUT_FRAMES_LISTS, exist_ok=True)

action_to_id = {}
current_action_id = 1
video_to_rows = defaultdict(list)

print("Starting elaboration...")

FIXED_ACTION_MAPPING = {
    "baseball_pitch": 1,
    "baseball_swing": 2,
    "golf_swing": 3,
    "squat": 4,
    "jumping_jacks": 5,
    "clean_and_jerk": 6,
    "bench_press": 7,
    "tennis_serve": 8,
    "pushup": 9,
    "situp": 10,
    "pullup": 11,
    "bowl": 12,
    "jump_rope": 13,
    "strum_guitar": 14,
    "tennis_forehand": 15
}

# Load data
for act_file in sorted(glob.glob(os.path.join(ACTION_DIR, "*_actions.json"))):
    basename = os.path.basename(act_file)

    match = re.search(r'(.*)_person_(\d+)_actions\.json', basename)
    if not match:
        continue

    video_id = match.group(1)
    person_id = match.group(2)
    person_key = f"person_{person_id}"

    src_frames = os.path.join(FRAMES_DIR, video_id)
    if not os.path.isdir(src_frames):
        continue

    det_folder = os.path.join(DETECTION_DIR, video_id)
    det_files = glob.glob(os.path.join(det_folder, "*.json"))
    if not det_files:
        continue
    det_file = det_files[0]

    with open(act_file, 'r') as f:
        actions_data = json.load(f)

    with open(det_file, 'r') as f:
        det_data = json.load(f)

    objects = det_data.get("detected_objects", {})
    if person_key not in objects:
        continue

    person_data = objects[person_key]
    if person_data.get("class_name") != "person":
        continue

    bboxes = person_data.get("bbox", [])
    max_frames = len(bboxes)

    action_frequencies = Counter()
    windows = []

    for frame_range, action_info in actions_data.items():
        act = action_info.get("action")
        if not act:
            continue

        rmatch = re.search(r'frames_(\d+)_to_(\d+)', frame_range)
        if not rmatch:
            continue

        start_f = int(rmatch.group(1))
        end_f = int(rmatch.group(2))

        action_frequencies[act] += 1
        windows.append((start_f, end_f, act))

    person_temp_rows = []

    for frame_idx in range(0, max_frames, 30):
        bbox = bboxes[frame_idx]
        if not bbox:
            continue

        candidate_actions = []
        for start_f, end_f, act in windows:
            if start_f <= frame_idx <= end_f:
                candidate_actions.append(act)

        if not candidate_actions:
            continue

        best_action = max(candidate_actions, key=lambda a: action_frequencies[a])

        if best_action not in FIXED_ACTION_MAPPING:
            continue

        act_id = FIXED_ACTION_MAPPING[best_action]
        x1, y1, x2, y2 = [format(coord, '.4f') for coord in bbox]
        timestamp = str(frame_idx // 30)

        row = f"{video_id},{timestamp},{x1},{y1},{x2},{y2},{act_id},{person_id}"
        person_temp_rows.append(row)

    if person_temp_rows:
        # Bufferizza le righe associandole al video
        video_to_rows[video_id].extend(person_temp_rows)

# Shuffling
valid_videos = sorted(video_to_rows.keys())

random.seed(42) # FIXED SEED FOR REPLICABILITY
random.shuffle(valid_videos)

num_videos = len(valid_videos)
train_split = int(num_videos * 0.8) # 80%
val_split = int(num_videos * 0.9)   # 10%

splits = {
    "train": valid_videos[:train_split],
    "val": valid_videos[train_split:val_split],
    "test": valid_videos[val_split:]
}

print(f"Video validi totali: {num_videos}")
print(f"Suddivisione: Train={len(splits['train'])}, Val={len(splits['val'])}, Test={len(splits['test'])}")

def extract_frame_number(filename):
    numbers = re.findall(r'\d+', filename)
    return int(numbers[0]) if numbers else 0

# Saving
for split_name, split_vids in splits.items():
    if not split_vids:
        continue

    split_annotations = []
    for vid in split_vids:
        split_annotations.extend(video_to_rows[vid])

    csv_path = os.path.join(OUT_ANNOTATIONS, f"ava_{split_name}_v2.2.csv")
    with open(csv_path, 'w') as f:
        f.write("\n".join(split_annotations) + "\n")

    list_path = os.path.join(OUT_FRAMES_LISTS, f"{split_name}.csv")
    with open(list_path, 'w') as f_list:

        for vid in split_vids:
            src_frames = os.path.join(FRAMES_DIR, vid)
            dst_frames = os.path.join(OUT_FRAMES, vid)

            if os.path.exists(src_frames) and not os.path.exists(dst_frames):
                try:
                    os.symlink(src_frames, dst_frames)
                except OSError:
                    shutil.copytree(src_frames, dst_frames)

            frame_files = [img for img in os.listdir(src_frames) if img.lower().endswith(('.jpg', '.jpeg', '.png'))]
            frame_files.sort(key=extract_frame_number)

            for i, frame_name in enumerate(frame_files, 1):
                path = f"{vid}/{frame_name}"
                dummy_label = "0"
                row = f"{vid} {vid} {i} {path} {dummy_label}\n"
                f_list.write(row)

pbtxt_path = os.path.join(OUT_ANNOTATIONS, "ava_action_list_v2.2.pbtxt")
with open(pbtxt_path, 'w') as f:
    for act_name, act_id in sorted(FIXED_ACTION_MAPPING.items(), key=lambda x: x[1]):
        f.write("label {\n")
        f.write(f'  name: "{act_name}"\n')
        f.write(f'  label_id: {act_id}\n')
        f.write('  label_type: 2\n')
        f.write("}\n")

print("Dataset generated and splitted.")


# DATASET AVA from GROUND TRUTH PENNACTION

PENNACTION_CSV_DIR = os.path.join(BASE_DIR, "PennAction")

PENNACTION_AVA_ROOT = os.path.join(BASE_DIR, "AVA_Dataset_PennAction")
PENN_OUT_ANNOTATIONS = os.path.join(PENNACTION_AVA_ROOT, "annotations")
PENN_OUT_FRAMES = os.path.join(PENNACTION_AVA_ROOT, "frames")
PENN_OUT_FRAMES_LISTS = os.path.join(PENNACTION_AVA_ROOT, "frame_lists")

os.makedirs(PENN_OUT_ANNOTATIONS, exist_ok=True)
os.makedirs(PENN_OUT_FRAMES, exist_ok=True)
os.makedirs(PENN_OUT_FRAMES_LISTS, exist_ok=True)

PENN_KEYFRAME_STRIDE = 30
PENN_PERSON_ID = "1"

penn_action_to_id = {}
penn_current_action_id = 1
penn_video_to_rows = defaultdict(list)


def get_frame_dimensions(frames_dir):
    """Returns (width, height) lfrom first frame."""
    frame_files = sorted(
        img for img in os.listdir(frames_dir)
        if img.lower().endswith(('.jpg', '.jpeg', '.png'))
    )
    if not frame_files:
        return None
    try:
        with Image.open(os.path.join(frames_dir, frame_files[0])) as im:
            return im.size  # (width, height)
    except Exception:
        return None


print("\Starting PennAction gt elaboration...")

for csv_path in sorted(glob.glob(os.path.join(PENNACTION_CSV_DIR, "*.csv"))):
    video_id = os.path.splitext(os.path.basename(csv_path))[0]

    src_frames = os.path.join(FRAMES_DIR, video_id)
    if not os.path.isdir(src_frames):
        continue

    frame_dims = get_frame_dimensions(src_frames)
    if frame_dims is None:
        continue
    frame_width, frame_height = frame_dims
    if frame_width <= 0 or frame_height <= 0:
        continue

    person_rows = []
    with open(csv_path, 'r', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                frame_id = int(row["frame_id"])
            except (ValueError, TypeError, KeyError):
                continue

            act = row.get("action")
            if not act:
                continue

            if (frame_id - 1) % PENN_KEYFRAME_STRIDE != 0:
                continue

            try:
                x1 = float(row["x_min"])
                y1 = float(row["y_min"])
                x2 = float(row["x_max"])
                y2 = float(row["y_max"])
            except (ValueError, TypeError, KeyError):
                continue

            if act not in FIXED_ACTION_MAPPING:
                continue

            act_id = FIXED_ACTION_MAPPING[act]
            timestamp = str((frame_id - 1) // PENN_KEYFRAME_STRIDE)

            x1_norm = min(max(x1 / frame_width, 0.0), 1.0)
            y1_norm = min(max(y1 / frame_height, 0.0), 1.0)
            x2_norm = min(max(x2 / frame_width, 0.0), 1.0)
            y2_norm = min(max(y2 / frame_height, 0.0), 1.0)

            row_str = (
                f"{video_id},{timestamp},"
                f"{format(x1_norm, '.4f')},{format(y1_norm, '.4f')},"
                f"{format(x2_norm, '.4f')},{format(y2_norm, '.4f')},"
                f"{act_id},{PENN_PERSON_ID}"
            )
            person_rows.append(row_str)

    if person_rows:
        penn_video_to_rows[video_id].extend(person_rows)

# Split 80/10/10
penn_valid_videos = sorted(penn_video_to_rows.keys())
random.seed(42)
random.shuffle(penn_valid_videos)

penn_num_videos = len(penn_valid_videos)
penn_train_split = int(penn_num_videos * 0.8)
penn_val_split = int(penn_num_videos * 0.9)

penn_splits = {
    "train": penn_valid_videos[:penn_train_split],
    "val": penn_valid_videos[penn_train_split:penn_val_split],
    "test": penn_valid_videos[penn_val_split:]
}

print(f"[PennAction] total: {penn_num_videos}")
print(f"[PennAction] Train={len(penn_splits['train'])}, "
      f"Val={len(penn_splits['val'])}, Test={len(penn_splits['test'])}")

for split_name, split_vids in penn_splits.items():
    if not split_vids:
        continue

    split_annotations = []
    for vid in split_vids:
        split_annotations.extend(penn_video_to_rows[vid])

    csv_out_path = os.path.join(PENN_OUT_ANNOTATIONS, f"ava_{split_name}_v2.2.csv")
    with open(csv_out_path, 'w') as f:
        f.write("\n".join(split_annotations) + "\n")

    list_path = os.path.join(PENN_OUT_FRAMES_LISTS, f"{split_name}.csv")
    with open(list_path, 'w') as f_list:
        for vid in split_vids:
            src_frames = os.path.join(FRAMES_DIR, vid)
            dst_frames = os.path.join(PENN_OUT_FRAMES, vid)

            if os.path.exists(src_frames) and not os.path.exists(dst_frames):
                try:
                    os.symlink(src_frames, dst_frames)
                except OSError:
                    shutil.copytree(src_frames, dst_frames)

            frame_files = [img for img in os.listdir(src_frames) if img.lower().endswith(('.jpg', '.jpeg', '.png'))]
            frame_files.sort(key=extract_frame_number)

            for i, frame_name in enumerate(frame_files, 1):
                path = f"{vid}/{frame_name}"
                dummy_label = "0"
                f_list.write(f"{vid} {vid} {i} {path} {dummy_label}\n")

pbtxt_path = os.path.join(PENN_OUT_ANNOTATIONS, "ava_action_list_v2.2.pbtxt")
with open(pbtxt_path, 'w') as f:
    for act_name, act_id in sorted(FIXED_ACTION_MAPPING.items(), key=lambda x: x[1]):
        f.write("label {\n")
        f.write(f'  name: "{act_name}"\n')
        f.write(f'  label_id: {act_id}\n')
        f.write('  label_type: 2\n')
        f.write("}\n")

print("Generated dataset AVA from PennAction.")

In [ ]:
# SPLITTING DATASET into SUBSETS
import os
import shutil
import random
from collections import defaultdict

# PATHS
ORIGINAL_AVA_ROOT = "/content/AVA_Dataset"
BASE_OUTPUT_DIR = "/content"

TRAIN_PERCENTAGES = [0.0625, 0.125, 0.25, 0.5, 0.75]
RANDOM_SEED = 42

IN_ANNOTATIONS = os.path.join(ORIGINAL_AVA_ROOT, "annotations")
IN_FRAMES_LISTS = os.path.join(ORIGINAL_AVA_ROOT, "frame_lists")
IN_FRAMES = os.path.join(ORIGINAL_AVA_ROOT, "frames")

train_anno_in = os.path.join(IN_ANNOTATIONS, "ava_train_v2.2.csv")
train_list_in = os.path.join(IN_FRAMES_LISTS, "train.csv")

random.seed(RANDOM_SEED)

video_to_action = {}
video_to_rows = defaultdict(list)

with open(train_anno_in, 'r') as f:
    for line in f:
        parts = line.strip().split(',')
        if len(parts) >= 8:
            vid_id = parts[0]
            action_id = parts[6]
            video_to_rows[vid_id].append(line)
            if vid_id not in video_to_action:
                video_to_action[vid_id] = action_id

action_to_videos = defaultdict(list)
for vid, act in video_to_action.items():
    action_to_videos[act].append(vid)

for act in action_to_videos:
    action_to_videos[act].sort()
    random.shuffle(action_to_videos[act])

with open(train_list_in, 'r') as f:
    train_list_lines = f.readlines()

val_test_videos = set()
for split in ["val", "test"]:
    list_path = os.path.join(IN_FRAMES_LISTS, f"{split}.csv")
    if os.path.exists(list_path):
        with open(list_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    val_test_videos.add(parts[0])

# Generating subset
for pct in TRAIN_PERCENTAGES:
    pct_str = str(pct * 100).replace('.', '_')
    print(f"\nGenerazione subset al {pct_str}%...")

    subset_root = os.path.join(BASE_OUTPUT_DIR, f"AVA_Dataset_Subset_{pct_str}")
    out_anno = os.path.join(subset_root, "annotations")
    out_lists = os.path.join(subset_root, "frame_lists")
    out_frames = os.path.join(subset_root, "frames")

    os.makedirs(out_anno, exist_ok=True)
    os.makedirs(out_lists, exist_ok=True)
    os.makedirs(out_frames, exist_ok=True)

    shutil.copy2(os.path.join(IN_ANNOTATIONS, "ava_action_list_v2.2.pbtxt"), out_anno)
    for split in ["val", "test"]:
        csv_in = os.path.join(IN_ANNOTATIONS, f"ava_{split}_v2.2.csv")
        if os.path.exists(csv_in):
            shutil.copy2(csv_in, out_anno)
        list_in = os.path.join(IN_FRAMES_LISTS, f"{split}.csv")
        if os.path.exists(list_in):
            shutil.copy2(list_in, out_lists)

    kept_train_videos = set()
    for act, videos in action_to_videos.items():
        num_keep = max(1, int(len(videos) * pct)) if len(videos) > 0 else 0
        kept_train_videos.update(videos[:num_keep])

    print(f"  - Video mantenuti nel Train: {len(kept_train_videos)}")

    with open(os.path.join(out_anno, "ava_train_v2.2.csv"), 'w') as f_out:
        for vid in kept_train_videos:
            for line in video_to_rows[vid]:
                f_out.write(line)

    with open(os.path.join(out_lists, "train.csv"), 'w') as f_out:
        for line in train_list_lines:
            parts = line.strip().split()
            if parts and parts[0] in kept_train_videos:
                f_out.write(line)

    videos_to_keep = kept_train_videos.union(val_test_videos)

    for vid in videos_to_keep:
        src_dir = os.path.join(IN_FRAMES, vid)
        dst_dir = os.path.join(out_frames, vid)

        if os.path.exists(src_dir) and not os.path.exists(dst_dir):
            try:
                os.symlink(src_dir, dst_dir)
            except OSError:
                shutil.copytree(src_dir, dst_dir)

print("\nSubset generated.")

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/AA-STAL/YOWOv2_finetuning/YOWOv2'

os.chdir(DRIVE_ROOT)
print(f"Directory: {os.getcwd()}")

In [ ]:
!pip install thop

Training 6.25%

In [ ]:
!python train.py \
  --cuda \
  --eval \
  -d ava_v2.2 \
  -v yowo_v2_tiny \
  --num_workers 2 \
  -r /content/drive/MyDrive/AA-STAL/YOWOv2_finetuning/YOWOv2/weights/yowov2_tiny_ava.pth \
  --root /content/ \
  --save_folder /content/drive/MyDrive/AA-STAL/YOWOv2_finetuning/YOWOv2/checkpoints_6_25/ \
  -bs 8 \
  -lr 0.0001 \
  --max_epoch 25 \
  --lr_epoch 15 20 \
  -ct 0.1 \
  --eval_epoch 1 \
  --freeze_backbone_3d

Training 12.5%

In [ ]:
!python train.py \
  --cuda \
  --eval \
  -d ava_v2.2 \
  -v yowo_v2_tiny \
  --num_workers 2 \
  -r /content/drive/MyDrive/AA-STAL/YOWOv2_finetuning/YOWOv2/weights/yowov2_tiny_ava.pth \
  --root /content/ \
  --save_folder /content/drive/MyDrive/AA-STAL/YOWOv2_finetuning/YOWOv2/checkpoints_12_5/ \
  -bs 8 \
  -lr 0.0001 \
  --max_epoch 20 \
  --lr_epoch 12 17 \
  -ct 0.1 \
  --eval_epoch 1 \
  --freeze_backbone_3d

Training 25%

In [ ]:
!python train.py \
  --cuda \
  --eval \
  -d ava_v2.2 \
  -v yowo_v2_tiny \
  --num_workers 2 \
  -r /content/drive/MyDrive/AA-STAL/YOWOv2_finetuning/YOWOv2/weights/yowov2_tiny_ava.pth \
  --root /content/ \
  --save_folder /content/drive/MyDrive/AA-STAL/YOWOv2_finetuning/YOWOv2/checkpoints_25/ \
  -bs 8 \
  -lr 0.0001 \
  --max_epoch 15 \
  --lr_epoch 9 13 \
  -ct 0.1 \
  --eval_epoch 1

Training 50%

In [ ]:
!python train.py \
  --cuda \
  --eval \
  -d ava_v2.2 \
  -v yowo_v2_tiny \
  --num_workers 2 \
  -r /content/drive/MyDrive/AA-STAL/YOWOv2_finetuning/YOWOv2/weights/yowov2_tiny_ava.pth \
  --root /content/ \
  --save_folder /content/drive/MyDrive/AA-STAL/YOWOv2_finetuning/YOWOv2/checkpoints_50/ \
  -bs 8 \
  -lr 0.0001 \
  --max_epoch 12 \
  --lr_epoch 8 11 \
  -ct 0.1 \
  --eval_epoch 1

Training 75%

In [ ]:
!python train.py \
  --cuda \
  --eval \
  -d ava_v2.2 \
  -v yowo_v2_tiny \
  --num_workers 2 \
  -r /content/drive/MyDrive/AA-STAL/YOWOv2_finetuning/YOWOv2/weights/yowov2_tiny_ava.pth \
  --root /content/ \
  --save_folder /content/drive/MyDrive/AA-STAL/YOWOv2_finetuning/YOWOv2/checkpoints_75/ \
  -bs 8 \
  -lr 0.0001 \
  --max_epoch 10 \
  --lr_epoch 7 9 \
  -ct 0.1 \
  --eval_epoch 1

Training 100%

In [ ]:
!python train.py \
  --cuda \
  --eval \
  -d ava_v2.2 \
  -v yowo_v2_tiny \
  --num_workers 2 \
  -r /content/drive/MyDrive/AA-STAL/YOWOv2_finetuning/YOWOv2/weights/yowov2_tiny_ava.pth \
  --root /content/ \
  --save_folder /content/drive/MyDrive/AA-STAL/YOWOv2_finetuning/YOWOv2/checkpoints_100/ \
  -bs 8 \
  -lr 0.0001 \
  --max_epoch 10 \
  --lr_epoch 5 8 \
  -ct 0.1 \
  --eval_epoch 1

Training over GT

In [ ]:
!python train.py \
  --cuda \
  --eval \
  -d ava_v2.2 \
  -v yowo_v2_tiny \
  --num_workers 2 \
  -r /content/drive/MyDrive/AA-STAL/YOWOv2_finetuning/YOWOv2/weights/yowov2_tiny_ava.pth \
  --root /content/ \
  --save_folder /content/drive/MyDrive/AA-STAL/YOWOv2_finetuning/YOWOv2/checkpoints_100_gt/ \
  -bs 8 \
  -lr 0.0001 \
  --max_epoch 15 \
  --lr_epoch 7 10 \
  -ct 0.1 \
  --eval_epoch 1

Evaluation

In [ ]:
!python eval.py \
    --cuda \
    -d ava_v2.2 \
    -v yowo_v2_tiny \
    -bs 4 \
    --weight /content/drive/MyDrive/AA-STAL/YOWOv2_finetuning/YOWOv2/checkpoints_100/ava_v2.2/yowo_v2_tiny/yowo_v2_tiny_epoch_9.pth \
    --root /content/ \
    -ct 0.1 \
    -nt 0.5

Test for process mining logs

In [ ]:
!python demo_process_mining.py \
  --cuda \
  -d ava_v2.2 \
  -v yowo_v2_tiny \
  --weight /content/drive/MyDrive/AA-STAL/YOWOv2_finetuning/YOWOv2/checkpoints_100/ava_v2.2/yowo_v2_tiny/yowo_v2_tiny_epoch_9.pth \
  --video /content/videoplayback.mp4 \
  --log_freq 16 \
  --save_folder /content/det_results/